# 14. ストリーミングの監視 - 動いていることと、追いつけていることは違う

最後のトピックです。

ここまでで、取り込み、変換、パイプライン、最適化、ジョブまでを見てきました。
残るのは **動かした後にどう見るか** です。

ストリーミングで一番怖いのは、落ちることではありません。落ちれば `13` の通知で気づけます。
怖いのは **落ちていないのに、だんだん遅れていく** ことです。

- 上流のデータが増えて、処理が追いつかなくなった
- 1バッチあたりの時間が少しずつ伸びている
- `07` のウォーターマークで、知らないうちに行が捨てられている

どれもエラーになりません。**見に行かないと分かりません。**

このノートブックで確かめること:

1. バッチごとの処理状況をどう読むか
2. 「追いつけているか」をどこで見るか
3. 継続的に監視するにはどうするか
4. 何を見て、何でアラートを出すか

**前提**: `00_setup` を実行済みであること。`02` と `07` を読んでいること。


## 準備


In [1]:
import json

from databricks.connect import DatabricksSession
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [2]:
CATALOG = "tech_survey"
TABLE = f"{CATALOG}.bronze.metrics_orders"
LANDING = f"/Volumes/{CATALOG}/ops/landing/14_metrics"
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/14_metrics"

## 1. バッチごとの記録を読む

ファイルを6個置いて、`cloudFiles.maxFilesPerTrigger` を2にします。
1回のマイクロバッチで2ファイルずつ処理するので、3バッチに分かれます。

`02` でトリガーを見たときと同じ作りです。今回は結果ではなく **記録のほう** を見ます。


In [3]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
for path in (LANDING, CHECKPOINT):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass

# 1ファイルにつき5件、6ファイル置く
for i in range(6):
    rows = []
    for j in range(5):
        rows.append({"order_id": i * 5 + j, "amount": (i + 1) * 100})

    dbutils.fs.put(f"{LANDING}/f{i}.json", "\n".join(json.dumps(r) for r in rows), True)

print("置いたファイル数:", len(dbutils.fs.ls(LANDING)))

置いたファイル数: 6


In [4]:
query = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .option("cloudFiles.maxFilesPerTrigger", 2)  # 1バッチ2ファイルまで
    .load(LANDING)
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)

query.awaitTermination()

print("バッチ数:", len(query.recentProgress))

バッチ数: 3


In [5]:
# バッチごとに、何行読んで何ミリ秒かかったかを見る
for p in query.recentProgress:
    print(f"batchId={p.batchId}  numInputRows={p.numInputRows}  durationMs={p.durationMs}")

batchId=0  numInputRows=10  durationMs={'triggerExecution': 9176, 'queryPlanning': 393, 'collectSourceMetrics': 31, 'getBatch': 242, 'commitOffsets': 151, 'postCommitBatchCallback': 571, 'addBatch': 4516, 'latestOffset': 1995, 'postAddBatchCallback': 612, 'commitBatch': 374, 'walCommit': 310}
batchId=1  numInputRows=10  durationMs={'triggerExecution': 9531, 'queryPlanning': 255, 'collectSourceMetrics': 3, 'getBatch': 39, 'commitOffsets': 142, 'postCommitBatchCallback': 550, 'addBatch': 4101, 'latestOffset': 3, 'postAddBatchCallback': 571, 'commitBatch': 581, 'walCommit': 230}
batchId=2  numInputRows=10  durationMs={'triggerExecution': 9898, 'queryPlanning': 301, 'collectSourceMetrics': 2, 'getBatch': 40, 'commitOffsets': 95, 'postCommitBatchCallback': 696, 'addBatch': 3577, 'latestOffset': 6, 'postAddBatchCallback': 491, 'commitBatch': 756, 'walCommit': 363}


`recentProgress` に、マイクロバッチ1回ぶんの記録が1つずつ入っています。
`02` で使ったのと同じものです。

見方のポイントは **バッチ単位で並べること** です。
合計の処理行数だけ見ていても、遅くなってきているかどうかは分かりません。

`durationMs` は内訳が入った辞書になっています。
`addBatch` (実際にデータを処理した時間) が大半を占めていれば素直な状態です。
`latestOffset` や `walCommit` が伸びているなら、処理そのものではなく
**ファイルの一覧取得やチェックポイントの書き込み** で待っていることになります。

`01` で触れたスモールファイル問題や、`10` の話がここに効いてきます。
ファイルが多いほど、一覧を取るだけで時間がかかるようになります。


## 2. 追いつけているかを見る

ここからが本題です。**処理した量ではなく、残っている量を見ます。**

取り込み元ごとの情報が `sources` に入っています。中身を丸ごと出してみます。


In [6]:
# ソース側の記録。キー名は取り込み元の種類によって変わるので、まず中身を見る
for p in query.recentProgress:
    print(f"--- batchId={p.batchId} ---")
    print("  description:", p.sources[0].description)
    print("  metrics    :", p.sources[0].metrics)

--- batchId=0 ---
  description: CloudFilesSource[/Volumes/tech_survey/ops/landing/14_metrics]
  metrics    : {'batchSizeNumFiles': '2', 'isBacklogComputationComplete': 'true', 'numFilesOutstanding': '4', 'numFilesSkippedMissing': '0', 'numBytesOutstanding': '636', 'batchSizeNumBytes': '308', 'numFilesProcessed': '2', 'numFilesUnknownState': '0', 'numFilesSkippedCorrupted': '0'}
--- batchId=1 ---
  description: CloudFilesSource[/Volumes/tech_survey/ops/landing/14_metrics]
  metrics    : {'batchSizeNumFiles': '2', 'isBacklogComputationComplete': 'true', 'numFilesOutstanding': '0', 'numFilesSkippedMissing': '0', 'numBytesOutstanding': '0', 'batchSizeNumBytes': '318', 'numFilesProcessed': '2', 'numFilesUnknownState': '0', 'numFilesSkippedCorrupted': '0'}
--- batchId=2 ---
  description: CloudFilesSource[/Volumes/tech_survey/ops/landing/14_metrics]
  metrics    : {'batchSizeNumFiles': '2', 'isBacklogComputationComplete': 'true', 'numFilesOutstanding': '0', 'numFilesSkippedMissing': '0', 'n

`metrics` の中に、**まだ処理していない量** を表す値が入っています。
Auto Loader なら未処理のファイル数やバイト数がここに出ます。

この値の読み方は「大きいかどうか」ではありません。**時間とともにどう動くか** です。

| 未処理量の動き | 意味 |
|---|---|
| 0 に近いまま | 追いつけている |
| 増えたり減ったり | 波はあるが処理しきれている |
| **増え続ける** | **追いつけていない。いつか破綻する** |

一番大事なのは3つ目です。処理は成功し続けるので **エラーは一切出ません。**
それでも遅れは広がり続け、気づいたときには数日ぶん溜まっている、という形になります。

`07` のウォーターマークと組み合わさると、もっと厄介です。
遅れているあいだに届いたデータが、処理される頃には水位より古くなって **捨てられます。**
遅れが遅れを呼ぶ形になります。


## 3. 状態の大きさを見る

`07` で `stateOperators` を見ました。ウィンドウ集計が抱えている行数が入っている場所です。

今回のストリームは **ただの追記** なので、ここは空のはずです。


In [7]:
# 状態を持たない処理では、stateOperators は空になる
for p in query.recentProgress:
    print(f"batchId={p.batchId}  stateOperators={p.stateOperators}")

batchId=0  stateOperators=[]
batchId=1  stateOperators=[]
batchId=2  stateOperators=[]


空でした。状態を持つ処理かどうかが、ここで分かります。

`07` のようなウィンドウ集計や、ストリーム同士の結合を行うと、ここに値が入ります。
見るべきなのは `numRowsTotal` (抱えている行数) です。

**増え続けているなら、状態が解放されていません。**
ウォーターマークを指定し忘れている、あるいは閾値が長すぎる、というのが典型的な原因です。
メモリやストレージを圧迫して、最後は処理が止まります。


## 4. 継続的に監視するには

ここまでは、実行が終わった後に `recentProgress` を手で読んできました。
学ぶにはこれでよいのですが、運用では **勝手に集まる** 必要があります。

方法が3つあります。

**1. `StreamingQueryListener`**

バッチが終わるたびに呼ばれる関数を登録しておき、そこで値を外へ送ります。

```python
class MyListener(StreamingQueryListener):
    def onQueryProgress(self, event):
        # event.progress に recentProgress と同じものが入る
        ...

spark.streams.addListener(MyListener())
```

本来はこれが正攻法です。
ただし **この環境では動かない可能性が高い** です。
`06` で見たように、このワークスペースではサーバー側でPythonを実行する仕組み
(サンドボックス) が壊れており、`foreachBatch` も Python UDF も失敗しました。
リスナーも同じ仕組みを使うため、同様に失敗すると考えられます。
**ここでは試していません。**

**2. LDP ならイベントログ**

`09` でやったように、パイプラインならイベントログをUnity Catalogのテーブルに出せます。
リスナーを書かなくても、SQLで追えるようになります。
自分でストリームを書くより、この点でも楽になります。

**3. ジョブの通知**

`13` で見た `on_failure` の通知です。
ただしこれは **落ちたときにしか飛びません。** 「遅れている」は拾えません。

遅れを検知したいなら、未処理量をどこかのテーブルに書き出しておいて、
それを見るジョブやアラートを別に用意することになります。


## 5. 何を見て、何で起こすか

見る値は増やせますが、**アラートにする値は絞ります。** 多いと誰も読まなくなります。

| 見るもの | どこ | 何が分かる |
|---|---|---|
| 未処理量 | `sources[].metrics` | 追いつけているか |
| バッチ所要時間 | `durationMs` | 処理が重くなっていないか |
| 処理行数 | `numInputRows` | 流量。急に0なら上流が止まった疑い |
| 状態の行数 | `stateOperators` の `numRowsTotal` | 状態が解放されているか |
| 捨てた行数 | `stateOperators` の `numRowsDroppedByWatermark` | 遅延データの取りこぼし (`07`) |

このうち **アラートにする価値があるのは、未処理量が増え続けていること** です。
他はだいたい、その結果として現れます。

もう1つ挙げるなら **処理行数が急に0になったこと** です。
ストリームは元気に動いているのに上流が止まっている、という状態を拾えます。
「エラーが出ていない」ことは「正常」の証明になりません。

この考え方は、ここまで何度も出てきました。

- `06` … `txnVersion` が合わないと、書き込みは黙って捨てられる
- `07` … 水位より古い行は、黙って捨てられる
- `09` … expectations の違反は、黙って記録されるだけ
- `12` … `rescue` モードの列は、黙って `_rescued_data` に入る

**黙って進むものは、こちらから見に行く。** 監視は、その仕組みを用意することです。


## 考えてみる

- 未処理量が増え続けていると分かったとき、まず何を疑いますか
- `numInputRows` が0のバッチが続いています。これは異常でしょうか
- 14本を終えて、`docs/etl_strategy.md` に何を書きますか


### 答え

**Q1. 追いつけていないとき**

疑う順番としては、次のようになります。

1. **上流が増えたのか** … そもそも流量が変わったのなら、処理側を増やすしかありません
2. **1バッチが重くなったのか** … `durationMs` の内訳を見ます。
   `addBatch` が伸びているなら処理そのもの、`latestOffset` が伸びているならファイルの一覧取得です
3. **ファイルが増えすぎていないか** … `10` のスモールファイル問題。
   一覧を取るだけで時間がかかるようになります
4. **状態が膨らんでいないか** … `stateOperators` を見ます。`07` のウォーターマーク設定が原因のことがあります

打ち手としては、`maxFilesPerTrigger` で1バッチの量を調整する、
`10` の `OPTIMIZE` でファイルをまとめる、といったあたりになります。

**Q2. `numInputRows` が0のバッチ**

**それだけでは判断できません。** 新しいデータが無ければ0になるのは正常です。

異常かどうかは **上流の事情を知らないと決められません。**
5分おきに必ず届くはずのデータが1時間0なら異常ですし、
1日1回しか届かないデータなら0が続くのが普通です。

つまり監視の閾値は、システムの側ではなく **業務の側から決まります。**
「何分来なかったらおかしいのか」を、データを出す側と合意しておく必要があります。

**Q3. `docs/etl_strategy.md` に何を書くか**

このリポジトリの出発点は「メダリオン構成の日次更新ETL戦略を立てる」ことでした。
14本で材料は揃ったので、次のような判断を書き残す形になります。

- **各層の更新方式** … Bronzeは追記 (`01`)、Silverは `replaceWhere` か `mergeInto` (`04` `05`)、
  Goldはマテリアライズドビューか集計テーブル (`08`)
- **Jobs方式かLDP方式か** … テーブルを作る部分はLDP、前後の段取りはジョブ (`08` `13`)
- **冪等性の担保** … どの層で何を使うか (`06`)
- **品質チェックの置き場所** … Bronzeでは弾かず、Silverで `drop` (`09`)
- **保持期間と最適化** … `VACUUM` の保持期間、`OPTIMIZE` を任せるか (`10` `11`)
- **監視** … 未処理量のアラート (`14`)

「どれが正しいか」ではなく、**この環境とこの要件では何を選ぶか** を書くのが目的です。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# for path in (LANDING, CHECKPOINT):
#     try:
#         dbutils.fs.rm(path, True)
#     except NotFound:
#         pass